# TIFM QLoRA Fine-Tuning on Kaggle

Fine-tune Qwen 2.5 Instruct on TIFM telecom analytics Q&A dataset.
Uses T4/P100 GPU (16GB VRAM). Runs in ~2-3 hours.

**Output**: Download `tifm_lora_output.zip` and extract to your `backend/tifm_lora_output/` folder.

In [ ]:
# ── Install dependencies ──
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl

In [ ]:
# ── Import modules ──
import json
import random
import itertools
import os
from datetime import datetime, timedelta
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)

In [ ]:
# ── Configuration ──
# Change to 'Qwen/Qwen2.5-3B-Instruct' if you want faster training
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = '/kaggle/working/tifm_lora_output'

# QLoRA
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

# Training — T4/P100 16GB can handle batch_size=2, seq_len=2048
TRAINING_ARGS = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=200,
    max_steps=2000,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=20,
    save_steps=500,
    eval_strategy='no',
    output_dir=OUTPUT_DIR,
    optim='paged_adamw_8bit',
    save_total_limit=2,
    report_to='none',
    max_seq_length=2048,
    gradient_checkpointing=True,
)

print('Configuration loaded.')

In [ ]:
# ── Generate training data ──
print('Generating training dataset...')

SYSTEM_PROMPT = (
    'You are a Telecom Intelligence Assistant \u2014 an expert digital forensics investigator '
    'trained to analyze Call Detail Records (CDR), IP Detail Records (IPDR), and TIFM '
    'multi-agent analytics. Your role is to interpret structured analytics data and provide '
    'clear, evidence-driven answers to investigation questions. Always cite specific metrics, '
    'highlight anomalies, and assess confidence levels. Use professional forensics language.'
)

SUBJECT_POOL = [
    '10.1.1.32', '10.1.6.22', '1.1.1.1', '10.1.3.15',
    '172.16.0.5', '192.168.1.100', '10.0.0.45', '172.18.2.10',
    '+915555555501', '+915555555502', '+915555555503', '+915555555504',
    '+910123456789', '+910987654321', '+917777777701', '+917777777702',
]

IMSI_POOL = [
    '40412901338353', '40478177244239', '40447935789865', '40496469651729',
    '40478537279135', '40469464571817', '40496234993475', '40478438683251',
]

IMEI_POOL = [
    '3539385479847', '3594906782660', '3538066791933', '3533951383393',
    '3544160389756', '3514752700207', '3559957213856', '3537354091810',
]

TOWERS = ['TWR_DEL_01', 'TWR_DEL_02', 'TWR_DEL_03', 'TWR_MUM_01', 'TWR_MUM_02']

def pick(arr):
    return arr[random.randint(0, len(arr) - 1)]

def synthetic_analytics(subject_count=4, app_count=3, meeting_count=2,
                         sim_swap_count=0, device_change_count=0,
                         burner_scores=None, roles=None, communities=None):
    burner_scores = burner_scores or [5, 10, 15, 8]
    roles = roles or ['Kingpin / Coordinator', 'Hub Node / Lieutenant', 'Member', 'Bridge Node / Broker']
    subjects = random.sample(SUBJECT_POOL, min(subject_count, len(SUBJECT_POOL)))
    network_nodes = {}
    for i, sub in enumerate(subjects):
        network_nodes[sub] = {
            'degree_centrality': round(random.uniform(0.1, 0.9), 3),
            'betweenness_centrality': round(random.uniform(0.05, 0.7), 3),
            'inferred_role': roles[i] if i < len(roles) else 'Member',
        }
    identity = {}
    for i, sub in enumerate(subjects):
        sim_swaps, device_changes = [], []
        base_imsi = IMSI_POOL[i % len(IMSI_POOL)]
        base_imei = IMEI_POOL[i % len(IMEI_POOL)]
        for s in range(sim_swap_count if i == 0 else 0):
            sim_swaps.append({
                'timestamp': (datetime.now() - timedelta(days=s * 2)).isoformat(),
                'imei': base_imei,
                'old_imsi': f'{IMSI_POOL[(i + s) % len(IMSI_POOL)]}',
                'new_imsi': f'{IMSI_POOL[(i + s + 1) % len(IMSI_POOL)]}',
                'type': 'SIM Swap', 'confidence': 'High',
            })
        for d in range(device_change_count if i == 1 else 0):
            device_changes.append({
                'timestamp': (datetime.now() - timedelta(days=d * 3)).isoformat(),
                'imsi': base_imsi, 'old_imei': base_imei,
                'new_imei': IMEI_POOL[(i + d + 1) % len(IMEI_POOL)],
                'type': 'Device Change', 'confidence': 'High',
            })
        identity[sub] = {
            'unique_identities_count': 1 + len(sim_swaps) + len(device_changes),
            'unique_pairs': [f'IMSI: {base_imsi} / IMEI: {base_imei}'],
            'sim_swaps': sim_swaps, 'device_changes': device_changes,
            'burner_score': burner_scores[i] if i < len(burner_scores) else 5,
            'is_suspected_burner': (burner_scores[i] if i < len(burner_scores) else 5) > 30,
        }
    movement = {}
    for i, sub in enumerate(subjects):
        movement[sub] = {
            'home_tower': TOWERS[0], 'work_tower': TOWERS[1],
            'total_towers_visited': random.randint(1, len(TOWERS)),
            'towers': TOWERS[:random.randint(1, len(TOWERS))],
            'mobility_index': pick(['High', 'Medium', 'Low']),
        }
    meetings = []
    meeting_pairs = list(itertools.combinations(subjects[:min(4, len(subjects))], 2))
    for m in range(min(meeting_count, len(meeting_pairs))):
        a, b = meeting_pairs[m]
        base_time = datetime.now() - timedelta(days=m)
        meetings.append({
            'tower_id': TOWERS[m % len(TOWERS)],
            'subject_a': a, 'subject_b': b,
            'time_a': base_time.isoformat(),
            'time_b': (base_time + timedelta(minutes=random.randint(5, 55))).isoformat(),
            'gap_minutes': round(random.uniform(3, 55), 2),
        })
    apps = ['WhatsApp', 'Telegram', 'Signal', 'VPN', 'Web']
    attribution = {}
    for app in apps[:app_count]:
        attribution[app] = {
            'count': random.randint(5, 80),
            'confidence': random.randint(70, 98),
            'evidence': [f'Port match: {random.choice([443, 3478, 5222, 8443])}',
                          f'Protocol: {pick(["TCP", "UDP"])}'],
        }
    comm_list = []
    if communities is None:
        mid = len(subjects) // 2
        if mid > 0:
            comm_list.append({'id': 1, 'members': subjects[:mid]})
        if len(subjects) > mid:
            comm_list.append({'id': 2, 'members': subjects[mid:]})
    else:
        comm_list = [{'id': i + 1, 'members': c} for i, c in enumerate(communities)]
    return {
        'attribution': attribution,
        'identity': identity,
        'movement': {'subjects_movement': movement, 'detected_meetings': meetings},
        'network': {
            'nodes_count': len(subjects), 'edges_count': random.randint(10, 40),
            'network_density': round(random.uniform(0.05, 0.6), 3),
            'centrality_metrics': network_nodes, 'communities': comm_list,
        },
    }

def fmt_analytics(analytics):
    return json.dumps(analytics, indent=2)

def make_example(analytics, question, answer):
    return {
        'system': SYSTEM_PROMPT,
        'user': f'TIFM Analytics:\n{fmt_analytics(analytics)}\n\nInvestigator Question: {question}',
        'assistant': answer,
    }

examples = []
COUNT = 200  # examples per question type

# Type 1: Subject role analysis
for i in range(COUNT):
    role = pick(['Kingpin / Coordinator', 'Hub Node / Lieutenant', 'Bridge Node / Broker', 'Member'])
    analytics = synthetic_analytics(subject_count=6, roles=[role, 'Member', 'Member', 'Bridge Node / Broker', 'Hub Node / Lieutenant', 'Member'])
    subjects = list(analytics['network']['centrality_metrics'].keys())
    sub = subjects[i % len(subjects)]
    role = analytics['network']['centrality_metrics'][sub]['inferred_role']
    q = f'What is the role of subject {sub} in this communication network?'
    a = (f'Based on the TIFM network centrality analysis, subject **{sub}** has a degree centrality '
         f'of **{analytics["network"]["centrality_metrics"][sub]["degree_centrality"]}** and a betweenness '
         f'centrality of **{analytics["network"]["centrality_metrics"][sub]["betweenness_centrality"]}**. '
         f'Their inferred role is **{role}**. ')
    if 'Kingpin' in role:
        a += 'This subject controls a significant portion of the communication flow. Interdiction of this node would maximally disrupt the network.'
    elif 'Hub' in role:
        a += 'This subject serves as a lieutenant or middle-manager. Monitoring their communications may reveal leadership instructions.'
    elif 'Bridge' in role:
        a += 'This subject is a broker connecting different sub-networks. They may be the only link between two otherwise separate groups.'
    else:
        a += 'This subject is a regular network member. Their removal would have limited structural impact.'
    examples.append(make_example(analytics, q, a))

# Type 2: Identity / SIM swap analysis
for i in range(COUNT):
    has_swaps = i % 2 == 0
    has_dev = i % 3 == 0
    analytics = synthetic_analytics(subject_count=6, sim_swap_count=3 if has_swaps else 0,
        device_change_count=2 if has_dev else 0, burner_scores=[45 if has_swaps else 10, 5, 5, 12, 8, 3])
    subjects = list(analytics['identity'].keys())
    sub = subjects[0]
    id_data = analytics['identity'][sub]
    q = f'Analyze the identity profile of subject {sub}. Are there SIM swaps, device changes, or burner indicators?'
    if has_swaps:
        swaps_d = '; '.join(f'on {s["timestamp"]}: IMSI changed from {s["old_imsi"]} to {s["new_imsi"]} (IMEI: {s["imei"]})' for s in id_data['sim_swaps'])
        a = (f'Subject {sub} shows clear identity evasion behavior:\n\n'
             f'- **SIM Swaps detected**: {len(id_data["sim_swaps"])} instances \u2014 {swaps_d}\n'
             f'- **Burner Score**: {id_data["burner_score"]}/100 (threshold > 30 is suspicious)\n'
             f'- **Unique identity pairs**: {id_data["unique_identities_count"]}\n\n'
             'Multiple SIM swaps on the same device indicate the subject is rotating subscriber identities. Recommended: correlate with known events.')
        if has_dev:
            a += f'\n\nAdditionally, {len(id_data["device_changes"])} device changes were detected. The subject is also swapping handsets.'
    else:
        a = (f'Subject {sub} has a clean identity profile: no SIM swaps, no device changes, burner score {id_data["burner_score"]}/100. '
             'Single identity pair in use. No evidence of deliberate identity obfuscation.')
    examples.append(make_example(analytics, q, a))

# Type 3: Meeting analysis
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, meeting_count=2 + i % 3)
    meetings = analytics['movement']['detected_meetings']
    if not meetings:
        continue
    m = meetings[0]
    q = f'Explain the meeting between {m["subject_a"]} and {m["subject_b"]}. What does the evidence show?'
    a = (f'A meeting was detected between **{m["subject_a"]}** and **{m["subject_b"]}** at tower **{m["tower_id"]}** '
         f'with a gap of only **{m["gap_minutes"]} minutes**. {m["subject_a"]} at {m["time_a"]}, {m["subject_b"]} at {m["time_b"]}.\n\n'
         '**Significance**: A sub-1-hour co-location at the same cell tower is highly suggestive of an in-person meeting. '
         'The short time delta makes coincidental co-location unlikely.\n\n'
         '**Confidence**: Medium-High. Recommended: cross-reference meeting time with other intelligence sources.')
    examples.append(make_example(analytics, q, a))

# Type 4: Movement patterns
for i in range(COUNT):
    mobility = pick(['High', 'Medium', 'Low'])
    analytics = synthetic_analytics(subject_count=6)
    subjects = list(analytics['movement']['subjects_movement'].keys())
    sub = subjects[i % len(subjects)]
    analytics['movement']['subjects_movement'][sub] = {
        'home_tower': 'TWR_DEL_01', 'work_tower': 'TWR_DEL_02',
        'total_towers_visited': 5 if mobility == 'High' else 2,
        'towers': ['TWR_DEL_01', 'TWR_DEL_02', 'TWR_DEL_03', 'TWR_MUM_01'][:5 if mobility == 'High' else 2],
        'mobility_index': mobility,
    }
    q = f'Describe the movement and activity patterns of {sub}.'
    a = f'Subject {sub} demonstrates a **{mobility.lower()} mobility profile** with {analytics["movement"]["subjects_movement"][sub]["total_towers_visited"]} distinct towers visited.\n'
    if mobility == 'High':
        a += 'The subject moves between multiple towers across different sectors, suggesting a peripatetic role requiring travel.'
    elif mobility == 'Medium':
        a += 'The subject operates primarily between a home and work location with occasional movement to other areas.'
    else:
        a += 'The subject is highly localized, operating primarily from a single tower area.'
    examples.append(make_example(analytics, q, a))

# Type 5: Full report (condensed to save context)
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, app_count=4, meeting_count=3,
        sim_swap_count=1 if i % 2 == 0 else 0, device_change_count=1 if i % 3 == 0 else 0,
        burner_scores=[48, 12, 7, 22, 5, 35],
        roles=['Kingpin / Coordinator', 'Hub Node / Lieutenant', 'Member', 'Bridge Node / Broker', 'Member', 'Hub Node / Lieutenant'],
        communities=[SUBJECT_POOL[:3], SUBJECT_POOL[3:6]])
    q = 'Generate a comprehensive investigation report based on the TIFM analytics.'
    comms = analytics['network']['communities']
    a = f'# TIFM Investigation Report\n\n## 1. Network Overview\nThe network has {analytics["network"]["nodes_count"]} nodes and {analytics["network"]["edges_count"]} edges.\n\n## 2. Community Structure\n'
    for c in comms:
        a += f'- Community {c["id"]}: {", ".join(c["members"])}\n'
    a += '\n## 3. Key Roles\n'
    for sub, m in sorted(analytics['network']['centrality_metrics'].items(), key=lambda x: x[1]['degree_centrality'], reverse=True):
        a += f'- {sub}: {m["inferred_role"]} (degree={m["degree_centrality"]})\n'
    burner_count = sum(1 for v in analytics['identity'].values() if v['is_suspected_burner'])
    a += f'\n## 4. Identity\nSuspected burners: {burner_count}\n'
    for sub, v in analytics['identity'].items():
        if v['sim_swaps'] or v['device_changes']:
            a += f'- {sub}: {len(v["sim_swaps"])} swaps, {len(v["device_changes"])} changes, score {v["burner_score"]}\n'
    a += f'\n## 5. Meetings\n{len(analytics["movement"]["detected_meetings"])} co-location events.\n'
    a += '\n## 6. Recommendations\n1. Priority surveillance on highest-centrality node\n2. Investigate SIM swap timing\n3. Physical surveillance at meeting tower locations'
    examples.append(make_example(analytics, q, a))

# Type 6: Anomaly analysis
for i in range(COUNT):
    has_anomaly = i % 2 == 0
    analytics = synthetic_analytics(subject_count=6, sim_swap_count=2 if has_anomaly else 0,
        device_change_count=1 if has_anomaly else 0, burner_scores=[60 if has_anomaly else 8, 5, 5, 15, 3, 10],
        meeting_count=3 if has_anomaly else 0)
    q = 'What stands out in this data? Are there any suspicious patterns or anomalies?'
    if has_anomaly:
        a = ('Several significant findings require attention:\n\n'
             f'1. Identity manipulation: SIM swaps detected with burner score of 60.\n'
             f'2. Physical meetings: {len(analytics["movement"]["detected_meetings"])} co-location events.\n'
             '3. Network structure: Communities with clear hierarchy and kingpin bridging clusters.')
    else:
        a = 'The data presents a routine communication network with no immediate red flags. Standard monitoring recommended.'
    examples.append(make_example(analytics, q, a))

# Types 7-8: Context chips (condensed)
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, sim_swap_count=1 if i % 2 == 0 else 0,
        burner_scores=[38 if i % 2 == 0 else 5, 5, 5, 10, 3, 8], meeting_count=i % 3)
    subjects = list(analytics['identity'].keys())
    sub = subjects[i % len(subjects)]
    role = analytics['network']['centrality_metrics'][sub]['inferred_role']
    id_data = analytics['identity'][sub]
    move_data = analytics['movement']['subjects_movement'].get(sub, {})
    id_status = 'Suspected burner' if id_data['is_suspected_burner'] else 'Clean profile'
    q1 = f'Explain what we know about subject {sub} and assess their role in the network.'
    a1 = (f'Subject **{sub}** profile: Network role: **{role}**, Identity: {id_status}, '
          f'Movement: {move_data.get("mobility_index", "Unknown")} mobility, '
          f'Towers: {move_data.get("total_towers_visited", 0)}. This subject plays a {role.lower()} role.')
    examples.append(make_example(analytics, q1, a1))
    q2 = f'Explain the tower movement pattern of {sub}.'
    towers_visited = move_data.get('towers', [])
    a2 = (f'Subject **{sub}** visited **{move_data.get("total_towers_visited", 0)}** towers '
          f'(mobility: {move_data.get("mobility_index", "Unknown")}). '
          f'Towers: {", ".join(towers_visited) if towers_visited else "None"}.')
    examples.append(make_example(analytics, q2, a2))

# Type 8: Meeting evidence
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, meeting_count=2 + i % 3)
    meetings = analytics['movement']['detected_meetings']
    if not meetings:
        continue
    m = meetings[0]
    q = f'What is the evidence for the meeting between {m["subject_a"]} and {m["subject_b"]} at {m["tower_id"]}?'
    a = (f'## Meeting Evidence Report\n\nSubjects: {m["subject_a"]} and {m["subject_b"]}\n'
         f'Location: Tower {m["tower_id"]}\n'
         f'Timeline: {m["subject_a"]} at {m["time_a"]}, {m["subject_b"]} at {m["time_b"]} (gap: {m["gap_minutes"]} min)\n\n'
         'Evidence:\n1. Both subjects connected to the same cell tower within a short window\n'
         f'2. Temporal gap of {m["gap_minutes"]} minutes below 60-min threshold\n'
         '3. Subjects have communication edges in CDR data\n\nConfidence: Medium-High (80-85%).')
    examples.append(make_example(analytics, q, a))

# Type 9: Schema boundary awareness
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, app_count=3)
    subjects = list(analytics['identity'].keys())
    sub = subjects[i % len(subjects)]
    app_name = pick(list(analytics['attribution'].keys()))
    q = f'What {app_name} activity does subject {sub} show?'
    a = (f'The TIFM analytics `attribution` section provides aggregate application statistics across all subjects. '
         f'It does **not** provide per-subject breakdowns. The total {app_name} count is '
         f'{analytics["attribution"][app_name]["count"]} sessions (confidence: {analytics["attribution"][app_name]["confidence"]}%). '
         'To determine per-subject activity, examine the raw IPDR timeline filtered by this subject.')
    examples.append(make_example(analytics, q, a))

# Type 10: Missing sections
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, app_count=0 if i % 2 == 0 else 3,
        meeting_count=0 if i % 3 == 0 else 2, sim_swap_count=0 if i % 4 == 0 else 1)
    if i % 2 == 0:
        q = 'What applications are being used?'
        a = 'No application attribution data is available. Either no IPDR records were provided or no known application signatures matched.'
        examples.append(make_example(analytics, q, a))
    if i % 3 == 0:
        q = 'Were any physical meetings detected?'
        a = 'No physical co-location events detected. No two subjects connected from the same tower within the detection window.'
        examples.append(make_example(analytics, q, a))
    if i % 4 == 0:
        q = 'Are there any SIM swaps or device changes?'
        a = 'No SIM swaps or device changes detected. All subjects show consistent identity usage.'
        examples.append(make_example(analytics, q, a))

# Type 11: Physical meeting vs app session
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, meeting_count=2, app_count=3)
    meetings = analytics['movement']['detected_meetings']
    if not meetings:
        continue
    m = meetings[i % len(meetings)]
    q = f'I see {m["subject_a"]} and {m["subject_b"]} communicated via WhatsApp. Is that the meeting you detected?'
    a = (f'No \u2014 the detected meeting between {m["subject_a"]} and {m["subject_b"]} refers to a physical in-person meeting, '
         f'not an app session. Both subjects connected to tower {m["tower_id"]} within {m["gap_minutes"]} minutes. '
         'App attribution (WhatsApp, etc.) tracks IP-level digital communication, not physical proximity. '
         'These are distinct signals that should be cross-referenced.')
    examples.append(make_example(analytics, q, a))

# Type 12: Confidence caveats
for i in range(COUNT):
    low = i % 2 == 0
    confs = {'WhatsApp': 65, 'Telegram': 58, 'Signal': 72} if low else {'WhatsApp': 95, 'Telegram': 88, 'Signal': 91}
    analytics = synthetic_analytics(subject_count=6, app_count=3)
    for app, c in confs.items():
        if app in analytics['attribution']:
            analytics['attribution'][app]['confidence'] = c
    app_name = pick(list(analytics['attribution'].keys()))
    conf = analytics['attribution'][app_name]['confidence']
    q = f'How confident is the {app_name} attribution?'
    a = f'The {app_name} attribution confidence level is **{conf}%**. '
    if conf < 75:
        a += 'This is low-to-moderate confidence. Attribution is based on supporting indicators but not definitive. Treat as an indicator, not proof.'
    elif conf < 85:
        a += 'This is moderate-to-high confidence. Patterns are reasonably consistent with {app_name} but consider VPN/proxy possibilities.'
    else:
        a += 'This is high confidence. Multiple independent indicators align. Attribution is reliable for investigative purposes.'
    examples.append(make_example(analytics, q, a))

# Type 13: Burner score explanation
for i in range(COUNT):
    high = i % 2 == 0
    score = pick([45, 55, 68, 72]) if high else pick([5, 8, 12, 18])
    analytics = synthetic_analytics(subject_count=6, sim_swap_count=3 if high else 0,
        burner_scores=[score, 5, 5, 12, 8, 3])
    subjects = list(analytics['identity'].keys())
    sub = subjects[0]
    id_data = analytics['identity'][sub]
    q = f'Subject {sub} has a burner score of {id_data["burner_score"]}. What does this mean?'
    if high:
        a = (f'Score {id_data["burner_score"]}/100 exceeds threshold 30, flagging as suspected burner. '
             f'{id_data["unique_identities_count"]} identity pairs, {len(id_data["sim_swaps"])} SIM swap(s). '
             'Indicates deliberate identity rotation for operational security.')
    else:
        a = f'Score {id_data["burner_score"]}/100 is below threshold 30. Single identity pair, no swaps. Normal device usage.'
    examples.append(make_example(analytics, q, a))

# Type 14: Multi-section cross-reference
for i in range(COUNT):
    analytics = synthetic_analytics(subject_count=6, sim_swap_count=1 if i % 2 == 0 else 0,
        meeting_count=1 + i % 2, burner_scores=[48 if i % 2 == 0 else 8, 5, 12, 7, 3, 15])
    subjects = list(analytics['identity'].keys())
    sub = subjects[i % len(subjects)]
    id_data = analytics['identity'].get(sub, {})
    centrality = analytics['network']['centrality_metrics'].get(sub, {})
    q = f'Build a complete profile of subject {sub} by combining all available analytics sections.'
    a = (f'## Subject Profile: {sub}\nRole: {centrality.get("inferred_role", "Unknown")} '
         f'(deg={centrality.get("degree_centrality", "N/A")}). '
         f'Identity: {id_data.get("unique_identities_count", 0)} pairs, score {id_data.get("burner_score", "N/A")}/100 '
         f'{"(BURNER)" if id_data.get("is_suspected_burner") else "(clean)"}. '
         f'Movement: {len(analytics["movement"]["subjects_movement"].get(sub, {}).get("towers", []))} towers.')
    if id_data.get('is_suspected_burner') and 'Kingpin' in centrality.get('inferred_role', ''):
        a += ' HIGH PRIORITY: Kingpin with burner behavior.'
    examples.append(make_example(analytics, q, a))

# Type 15: Data completeness
for i in range(COUNT):
    has_cdr = i % 2 == 0
    has_ipdr = i % 3 == 0
    parts = []
    if not has_cdr:
        parts.append('no CDR')
    if not has_ipdr:
        parts.append('no IPDR')
    if not parts:
        parts.append('both CDR and IPDR')
    q = 'How complete is this data? What types of records are included?'
    a = f'The analytics include {', '.join(parts)}. '
    if not has_cdr:
        a += 'Without CDR: no voice call graph, no SMS patterns. '
    if not has_ipdr:
        a += 'Without IPDR: no app attribution, no traffic analysis. '
    a += 'TIFM analytics are derived only from uploaded records. Upload all available files for comprehensive analysis.'
    examples.append(make_example(analytics, q, a))

random.shuffle(examples)
print(f'Generated {len(examples)} training examples')

dataset_path = '/kaggle/working/tifm_train.jsonl'
with open(dataset_path, 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print(f'Saved to {dataset_path}')

In [ ]:
# ── Load model and train ──
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BNB_CONFIG,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LORA_CONFIG)
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

print('Loading dataset...')
dataset = load_dataset('json', data_files=dataset_path, split='train')

def format_batch(batch):
    texts = []
    for s, u, a in zip(batch['system'], batch['user'], batch['assistant']):
        msgs = [
            {'role': 'system', 'content': s},
            {'role': 'user', 'content': u},
            {'role': 'assistant', 'content': a},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {'text': texts}

dataset = dataset.map(format_batch, batched=True)
print(f'Dataset size: {len(dataset)}')

print('Starting training...')
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=TRAINING_ARGS,
)
trainer.train()

# Save LoRA adapter
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

In [ ]:
# ── Zip and prepare for download ──
import shutil

zip_path = '/kaggle/working/tifm_lora_output.zip'
shutil.make_archive('/kaggle/working/tifm_lora_output', 'zip', OUTPUT_DIR)
print(f'Created {zip_path}')
print('\nDownload instructions:')
print('1. Go to the Kaggle notebook "Data" tab')
print('2. Find tifm_lora_output.zip under "Output"')
print('3. Download and extract to your backend/tifm_lora_output/ folder')
print('4. Restart the backend server')